# Notebook para cargar ventas 2025 inferencia
Este notebook importa las librerías necesarias y carga el archivo de ventas de 2025 para inferencia en un DataFrame.

In [1]:
# Importar librerías necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import streamlit as st
import holidays

In [2]:
# Cargar archivo de ventas 2025 inferencia
df_path = '../data/raw/inferencia/ventas_2025_inferencia.csv'
inferencia_df = pd.read_csv(df_path)

In [3]:
# Visualizar las primeras filas del DataFrame
inferencia_df.head()

,fecha,producto_id,nombre,categoria,subcategoria,precio_base,es_estrella,unidades_vendidas,precio_venta,ingresos,Amazon,Decathlon,Deporvillage
0,2025-10-25,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,26.0,113.13,2941.38,89.51,113.43,104.78
1,2025-10-25,PROD_002,Adidas Ultraboost 23,Running,Zapatillas Running,135,True,27.0,141.89,3831.03,128.73,112.91,122.88
2,2025-10-25,PROD_003,Asics Gel Nimbus 25,Running,Zapatillas Running,85,False,5.0,85.79,428.95,84.28,74.51,85.57
3,2025-10-25,PROD_004,New Balance Fresh Foam X 1080v12,Running,Zapatillas Running,75,False,3.0,76.19,228.57,75.54,70.32,71.13
4,2025-10-25,PROD_005,Nike Dri-FIT Miler,Running,Ropa Running,35,False,3.0,35.48,106.44,33.84,31.32,34.41


## Preparación de inferencia_df para el modelo de forecasting

Se aplican exactamente las mismas transformaciones e ingeniería de variables que se hicieron sobre `df` en el notebook de entrenamiento:
1. Conversión de fecha
2. Variables de calendario
3. Lags y media móvil de 7 días
4. Variables de precio (descuento, precio competencia, ratio)
5. One-Hot Encoding de variables categóricas
6. Alineación de columnas con el DataFrame de entrenamiento
7. Filtrado a solo registros de noviembre
8. Guardado en `data/processed/`

In [4]:
# Cargar df de entrenamiento procesado como referencia de columnas
df_referencia = pd.read_csv('../data/processed/df.csv')
df_referencia['fecha'] = pd.to_datetime(df_referencia['fecha'])
print(f'Columnas en df de entrenamiento: {df_referencia.shape[1]}')
df_referencia.head()

Columnas en df de entrenamiento: 126


,fecha,producto_id,nombre,categoria,subcategoria,precio_base,es_estrella,unidades_vendidas,precio_venta,ingresos,...,subcategoria_h_Esterilla Yoga.1,subcategoria_h_Mancuernas Ajustables.1,subcategoria_h_Mochila Trekking.1,subcategoria_h_Pesa Rusa.1,subcategoria_h_Pesas Casa.1,subcategoria_h_Rodillera Yoga.1,subcategoria_h_Ropa Montaña.1,subcategoria_h_Ropa Running.1,subcategoria_h_Zapatillas Running.1,subcategoria_h_Zapatillas Trail.1
0,2021-10-25,PROD_008,Reebok Floatride Energy 5,Running,Zapatillas Running,65,False,2,66.60,133.20,...,False,False,False,False,False,False,False,False,True,False
1,2021-10-25,PROD_009,Bowflex SelectTech 552,Fitness,Mancuernas Ajustables,400,True,2,392.31,784.62,...,False,True,False,False,False,False,False,False,False,False
2,2021-10-25,PROD_010,Domyos BM900,Fitness,Banco Gimnasio,175,True,2,174.92,349.84,...,False,False,False,False,False,False,False,False,False,False
3,2021-10-25,PROD_011,Reebok Professional Deck,Fitness,Esterilla Fitness,45,False,3,44.32,132.96,...,False,False,False,False,False,False,False,False,False,False
4,2021-10-25,PROD_012,Domyos Kit Mancuernas 20kg,Fitness,Pesas Casa,55,False,3,55.05,165.15,...,False,False,False,False,True,False,False,False,False,False


In [5]:
# Pipeline completo de transformacion
# Siempre parte del CSV original: seguro para re-ejecucion sin reiniciar kernel.
#
# NOTA sobre lags: unidades_vendidas es NaN en noviembre (es el target a predecir).
# Los lags se calculan desde los 7 dias de octubre y se aplican como contexto
# historico fijo a todos los dias de noviembre (lag_1 = Oct 31, lag_7 = Oct 25).

import os

# --- Carga fresca del CSV ---
_raw = pd.read_csv('../data/raw/inferencia/ventas_2025_inferencia.csv')
_raw['fecha'] = pd.to_datetime(_raw['fecha'])

# 1. Variables de calendario
_raw['anno'] = _raw['fecha'].dt.year
_raw['mes'] = _raw['fecha'].dt.month
_raw['dia_mes'] = _raw['fecha'].dt.day
_raw['dia_semana'] = _raw['fecha'].dt.weekday
_raw['nombre_dia'] = _raw['fecha'].dt.day_name(locale='es_ES')
_raw['semana_anno'] = _raw['fecha'].dt.isocalendar().week
_raw['es_fin_de_semana'] = _raw['dia_semana'].isin([5, 6])
_spain_hol = holidays.country_holidays('ES', years=[_raw['fecha'].dt.year.max()])
_raw['es_festivo'] = _raw['fecha'].isin(_spain_hol)
_raw['es_black_friday'] = (
    (_raw['fecha'].dt.month == 11) &
    (_raw['fecha'].dt.weekday == 4) &
    ((_raw['fecha'] + pd.offsets.Week(1)).dt.month == 12)
)
_bf = _raw.loc[_raw['es_black_friday'], 'fecha']
_raw['es_cyber_monday'] = _raw['fecha'].isin(_bf + pd.Timedelta(days=3))
_raw['es_laborable'] = (~_raw['es_fin_de_semana'] & ~_raw['es_festivo'])
_raw['dia_anno'] = _raw['fecha'].dt.dayofyear
_raw['es_primer_dia_mes'] = (_raw['dia_mes'] == 1)
_raw['es_ultimo_dia_mes'] = (_raw['fecha'] == _raw['fecha'] + pd.offsets.MonthEnd(0))
_raw['trimestre'] = _raw['fecha'].dt.quarter
_raw['semestre'] = np.where(_raw['mes'] <= 6, 1, 2)
_raw = _raw.rename(columns={'anno': 'anno_', 'semana_anno': 'semana_anno_', 'dia_anno': 'dia_anno_'})
_raw = _raw.rename(columns={'anno_': 'año', 'semana_anno_': 'semana_año', 'dia_anno_': 'dia_año'})

# 2. Lags y ma7 calculados desde los dias de octubre (datos conocidos)
#    lag_1 = valor de Oct 31, lag_7 = valor de Oct 25, ma7 = media Oct 25-31
_oct = _raw[_raw['mes'] == 10].sort_values(['producto_id', 'fecha'])
_lag_rows = []
for prod_id, grp in _oct.groupby('producto_id'):
    vals = grp['unidades_vendidas'].values  # ordenado por fecha, mas reciente al final
    n = len(vals)
    row = {'producto_id': prod_id}
    for lag in range(1, 8):
        row[f'unidades_vendidas_lag_{lag}'] = vals[-lag] if n >= lag else np.nan
    row['unidades_vendidas_ma7'] = float(np.mean(vals[-7:])) if n >= 7 else np.nan
    _lag_rows.append(row)
_lag_df = pd.DataFrame(_lag_rows)

# 3. Filtrar noviembre y unir los lags
_nov = _raw[_raw['mes'] == 11].copy().reset_index(drop=True)
_nov = _nov.merge(_lag_df, on='producto_id', how='left')

# Eliminar registros con NaN en lags (productos sin suficiente historia en octubre)
_lag_cols = [f'unidades_vendidas_lag_{i}' for i in range(1, 8)] + ['unidades_vendidas_ma7']
_nov = _nov.dropna(subset=_lag_cols).reset_index(drop=True)
print(f'Registros noviembre tras lags: {len(_nov)}')

# 4. Variables de precio
_nov['descuento_porcentaje'] = ((_nov['precio_venta'] - _nov['precio_base']) / _nov['precio_base']) * 100
_compet = ['Amazon', 'Decathlon', 'Deporvillage']
_nov['precio_competencia'] = _nov[_compet].mean(axis=1)
_nov['ratio_precio'] = _nov['precio_venta'] / _nov['precio_competencia']
_nov = _nov.drop(columns=_compet)

# 5. One-Hot Encoding
_nov['nombre_h'] = _nov['nombre']
_nov['categoria_h'] = _nov['categoria']
_nov['subcategoria_h'] = _nov['subcategoria']
_nov = pd.get_dummies(_nov, columns=['nombre_h', 'categoria_h', 'subcategoria_h'],
                      prefix=['nombre_h', 'categoria_h', 'subcategoria_h'])

# 6. Alinear columnas OHE con el entrenamiento
_ohe_pfx = ('nombre_h_', 'categoria_h_', 'subcategoria_h_')
_ohe_ref = [c for c in df_referencia.columns if c.startswith(_ohe_pfx)]
_ohe_new = {col: pd.Series([0] * len(_nov), dtype='uint8') for col in _ohe_ref if col not in _nov.columns}
if _ohe_new:
    _nov = pd.concat([_nov, pd.DataFrame(_ohe_new)], axis=1)
_extra = [c for c in _nov.columns if c.startswith(_ohe_pfx) and c not in _ohe_ref]
if _extra:
    _nov = _nov.drop(columns=_extra)
print(f'Columnas OHE referencia: {len(_ohe_ref)} | inferencia: {sum(c.startswith(_ohe_pfx) for c in _nov.columns)}')

# 7. Asignar al DataFrame final
inferencia_df = _nov.copy()

# 8. Guardar
os.makedirs('../data/processed', exist_ok=True)
inferencia_df.to_csv('../data/processed/inferencia_df_transformado.csv', index=False)
print(f'Archivo guardado en ../data/processed/inferencia_df_transformado.csv')
print(f'Shape final: {inferencia_df.shape}')

Registros noviembre tras lags: 720
Columnas OHE referencia: 88 | inferencia: 88
Archivo guardado en ../data/processed/inferencia_df_transformado.csv
Shape final: (720, 125)


In [20]:
df_referencia.columns

Index(['fecha', 'producto_id', 'nombre', 'categoria', 'subcategoria',
       'precio_base', 'es_estrella', 'unidades_vendidas', 'precio_venta',
       'ingresos',
       ...
       'subcategoria_h_Esterilla Yoga.1',
       'subcategoria_h_Mancuernas Ajustables.1',
       'subcategoria_h_Mochila Trekking.1', 'subcategoria_h_Pesa Rusa.1',
       'subcategoria_h_Pesas Casa.1', 'subcategoria_h_Rodillera Yoga.1',
       'subcategoria_h_Ropa Montaña.1', 'subcategoria_h_Ropa Running.1',
       'subcategoria_h_Zapatillas Running.1',
       'subcategoria_h_Zapatillas Trail.1'],
      dtype='str', length=126)